In [1]:
from pyspark.sql.functions import col, count, sum, avg, round, when, lit, current_date, max, coalesce,date_add
from pyspark.sql.types import BooleanType, DateType
from delta.tables import DeltaTable

StatementMeta(, 3229e0eb-c8c0-4ebf-83fa-ab2014430ab6, 3, Finished, Available, Finished, False)

In [14]:
dim_customers = spark.read.format("delta").load(
    "abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/silver/customers"
)

dim_customers.select(
    "customer_id", "age", "gender", "country",
    "membership_tier", "registration_date",
    "newsletter_subscribed", "churned"
)\
.withColumn("effective_date", date_add(current_date(), -1))\
.withColumn("end_date", lit(None).cast("date"))\
.withColumn("is_current", lit(True))\
.write.format("delta")\
.save("abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/gold/dim_customers")

print(f"Total Day 1 Rows: {dim_customers.count()}")


StatementMeta(, 3229e0eb-c8c0-4ebf-83fa-ab2014430ab6, 16, Finished, Available, Finished, False)

Total Day 1 Rows: 7959


In [15]:
display(dim_customers)

StatementMeta(, 3229e0eb-c8c0-4ebf-83fa-ab2014430ab6, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b3e695e2-20ca-410b-a530-73f05ae0fe56)

In [16]:
dim_customer2 = spark.read.format("delta").load(
    "abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/silver/customers2"
)

dim_customer2 = dim_customer2.select(
    "customer_id", "age", "gender", "country",
    "membership_tier", "registration_date",
    "newsletter_subscribed", "churned"
)
print(f"Total Day 2 Rows: {dim_customer2.count()}")
display(dim_customer2.select(
    "customer_id", "country",
    "membership_tier", "churned"
))

StatementMeta(, 3229e0eb-c8c0-4ebf-83fa-ab2014430ab6, 18, Finished, Available, Finished, False)

Total Day 2 Rows: 10


SynapseWidget(Synapse.DataFrame, 4199dc3d-862c-42f3-96e2-6a21b3553d29)

In [5]:
display(
    spark.read.format("delta").load(
        "abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/gold/dim_customers"
    )
    .filter(col("customer_id").isin([
        "C00001","C00002","C00003","C00004","C00005",
        "C00006","C00007","C00008","C00009","C00010"
    ]))
    .select("customer_id","country","membership_tier",
            "newsletter_subscribed","churned","is_current")
    .orderBy("customer_id")
)

display(
    df_customer2.select(
        "customer_id","country","membership_tier",
        "newsletter_subscribed","churned"
    ).orderBy("customer_id")
)

StatementMeta(, 3229e0eb-c8c0-4ebf-83fa-ab2014430ab6, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5fa703c7-fa6f-45a2-8419-c7df64430aef)

SynapseWidget(Synapse.DataFrame, 98249626-f896-4c57-bbb6-491351cbd444)

In [17]:
delta_gold = DeltaTable.forPath(
    spark,
    "abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/gold/dim_customers"
)

# Get active gold records
df_active_gold = delta_gold.toDF()\
    .filter(col("is_current") == True)\
    .select(
        col("customer_id").alias("gold_id"),
        col("age").alias("gold_age"),
        col("country").alias("gold_country"),
        col("membership_tier").alias("gold_tier"),
        col("newsletter_subscribed").alias("gold_newsletter"),
        col("churned").alias("gold_churned")
    )

# Join Day 2 with Gold
df_joined = dim_customer2.join(
    df_active_gold,
    dim_customer2.customer_id == col("gold_id"),
    "left"
)

# Find NEW customers
df_new = df_joined\
    .filter(col("gold_id").isNull())\
    .select(dim_customer2["*"])\
    .withColumn("merge_key", col("customer_id"))

# Find CHANGED customers
df_changed = df_joined.filter(
    col("gold_id").isNotNull() & (
        (col("age") != col("gold_age")) |
        (col("country") != col("gold_country")) |
        (col("membership_tier") != col("gold_tier")) |
        (col("newsletter_subscribed") != col("gold_newsletter")) |
        (col("churned") != col("gold_churned"))
    )
).select(dim_customer2["*"])

print(f"New Customers Found:     {df_new.count()}")
print(f"Changed Customers Found: {df_changed.count()}")

print("\n=== CHANGED CUSTOMERS ===")
display(df_changed.select(
    "customer_id", "country",
    "membership_tier", "churned"
))

StatementMeta(, 3229e0eb-c8c0-4ebf-83fa-ab2014430ab6, 19, Finished, Available, Finished, False)

New Customers Found:     0
Changed Customers Found: 10

=== CHANGED CUSTOMERS ===


SynapseWidget(Synapse.DataFrame, aa01bec8-9cf8-44f8-9b8d-46df8d56cc11)

In [18]:
# STEP 5 FIX Cast everything to BOOLEAN 
from pyspark.sql.types import BooleanType, DateType
from pyspark.sql.functions import col, lit, current_date

if df_changed.count() > 0 or df_new.count() > 0:

    # Fix df_new types
    df_new_fixed = df_new\
        .withColumn("newsletter_subscribed",
            col("newsletter_subscribed").cast(BooleanType()))\
        .withColumn("churned",
            col("churned").cast(BooleanType()))

    # Track A → Close old records
    track_a = df_changed\
        .withColumn("merge_key", col("customer_id"))\
        .withColumn("newsletter_subscribed",
            col("newsletter_subscribed").cast(BooleanType()))\
        .withColumn("churned",
            col("churned").cast(BooleanType()))

    # Track B → Insert new records
    track_b = df_changed\
        .withColumn("merge_key", lit(None).cast("string"))\
        .withColumn("newsletter_subscribed",
            col("newsletter_subscribed").cast(BooleanType()))\
        .withColumn("churned",
            col("churned").cast(BooleanType()))

    # Combine staging
    staging = df_new_fixed\
        .unionByName(track_a)\
        .unionByName(track_b)

    print(f"Staging Rows: {staging.count()}")
    print("Staging Schema:")
    staging.printSchema()

    # Execute Merge
    delta_gold.alias("target").merge(
        staging.alias("source"),
        "target.customer_id = source.merge_key AND target.is_current = true"
    ).whenMatchedUpdate(set={
        "end_date":   current_date(),
        "is_current": lit(False)
    }).whenNotMatchedInsert(values={
        "customer_id":           "source.customer_id",
        "age":                   "source.age",
        "gender":                "source.gender",
        "country":               "source.country",
        "membership_tier":       "source.membership_tier",
        "registration_date":     "source.registration_date",
        "newsletter_subscribed": "source.newsletter_subscribed",
        "churned":               "source.churned",
        "effective_date":        current_date(),
        "end_date":              lit(None).cast("date"),
        "is_current":            lit(True)
    }).execute()

    print("Merge Complete!")

else:
    print("No changes found!")

StatementMeta(, 3229e0eb-c8c0-4ebf-83fa-ab2014430ab6, 20, Finished, Available, Finished, False)

Staging Rows: 20
Staging Schema:
root
 |-- customer_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- membership_tier: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- newsletter_subscribed: boolean (nullable = true)
 |-- churned: boolean (nullable = true)
 |-- merge_key: string (nullable = true)

Merge Complete!


In [19]:
#  Verify Final Results 
df_final = spark.read.format("delta").load(
    "abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/gold/dim_customers"
)

print("=== FINAL GOLD TABLE SUMMARY ===")
print(f"Total Rows:         {df_final.count()}")
print(f"Current Records:    {df_final.filter('is_current = true').count()}")
print(f"Historical Records: {df_final.filter('is_current = false').count()}")

print("\n=== ALL HISTORICAL RECORDS ===")
display(
    df_final
    .filter("is_current = false")
    .select(
        "customer_id", "country", "membership_tier",
        "newsletter_subscribed", "churned",
        "effective_date", "end_date", "is_current"
    )
    .orderBy("customer_id")
)

StatementMeta(, 3229e0eb-c8c0-4ebf-83fa-ab2014430ab6, 21, Finished, Available, Finished, False)

=== FINAL GOLD TABLE SUMMARY ===
Total Rows:         7969
Current Records:    7959
Historical Records: 10

=== ALL HISTORICAL RECORDS ===


SynapseWidget(Synapse.DataFrame, 8fad943a-559e-4085-af4d-faefc97dfe77)

In [22]:
display(df_final.orderBy("customer_id", "effective_date"))

StatementMeta(, 3229e0eb-c8c0-4ebf-83fa-ab2014430ab6, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3f90a64a-74dd-44c5-bd87-ea0154054a20)

In [25]:
# Read orders data
df_orders = spark.read.format("delta").load(
    "abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/silver/orders"
)

customer_metrics = df_orders.groupBy("customer_id").agg(
    count("order_id").alias("total_orders"),
    round(sum("total_amount_usd"), 2).alias("total_spent"),
    round(avg("total_amount_usd"), 2).alias("avg_order_value"),
    max("order_date").alias("last_order_date"),
    round(avg("customer_rating"), 2).alias("avg_rating"),
    count(when(col("returned") == True, 1)).alias("return_count")
)

# Join metrics with customer dim
fact_customer_metrics = customer_metrics.join(dim_customers, "customer_id")

# Replace Avg rating as 0 if not provided
fact_customer_metrics = fact_customer_metrics.withColumn(
    "avg_rating", coalesce(col("avg_rating"), lit(0.0))
)

fact_customer_metrics.write.mode("overwrite").format("delta").save(
    "abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/gold/fact_customer_metrics"
)

print(f"fact_customer_metrics: {fact_customer_metrics.count()} records")

StatementMeta(, 015e6b0b-e6e0-4175-afb9-15c3dc7e8695, 27, Finished, Available, Finished, False)

fact_customer_metrics: 7621 records


In [6]:
df_orders = spark.read.format("delta").load(
    "abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Files/silver/orders"
)

customer_metrics = df_orders.groupBy("customer_id").agg(
    count("order_id").alias("total_orders"),
    round(sum("total_amount_usd"), 2).alias("total_spent"),
    round(avg("total_amount_usd"), 2).alias("avg_order_value"),
    max("order_date").alias("last_order_date"),
    round(avg("customer_rating"), 2).alias("avg_rating"),
    count(when(col("returned") == True, 1)).alias("return_count")
)

# Join metrics with customer dim
fact_customer_metrics = customer_metrics.join(dim_customers, "customer_id")

# Replace Avg rating as 0 if not provided
fact_customer_metrics = fact_customer_metrics.withColumn(
    "avg_rating", coalesce(col("avg_rating"), lit(0.0))
)

# Write to TABLES (registered) not Files
fact_customer_metrics.write.mode("overwrite").format("delta").saveAsTable("gold.fact_customer_metrics", path=
    "abfss://my_workspace@onelake.dfs.fabric.microsoft.com/Ecommerce_Data.Lakehouse/Tables/gold/fact_customer_metrics"
)

StatementMeta(, 48024e63-ad23-4bef-b6f3-74bd13e9e2a5, 8, Finished, Available, Finished, False)

In [ ]:
spark.catalog.clearCache()
spark.stop()